In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src.preprocessing import clean_data

In [2]:
dev = pd.read_csv("../data/dev.csv")

print(dev.shape)

(120000, 11)


In [3]:
dev_clean = clean_data(dev.copy())

print(dev_clean.shape)

(120000, 15)


In [4]:
print(dev_clean.columns.tolist())

['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'missingindicator_age', 'missingindicator_MonthlyIncome', 'missingindicator_NumberOfDependents', 'late_payment_sentinel']


In [5]:
print("Original age zero count:", (dev["age"] == 0).sum())
print("Cleaned age zero count:", (dev_clean["age"] == 0).sum())

Original age zero count: 1
Cleaned age zero count: 0


In [6]:
print(dev_clean.isnull().sum())

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         1
NumberOfTime30-59DaysPastDueNotWorse      214
DebtRatio                                   0
MonthlyIncome                           23675
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                   214
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse      214
NumberOfDependents                       3128
missingindicator_age                        0
missingindicator_MonthlyIncome              0
missingindicator_NumberOfDependents         0
late_payment_sentinel                       0
dtype: int64


In [7]:
late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for col in late_cols:
    print(f"\n{col}")
    print("Original sentinel count:", dev[col].isin([96, 98]).sum())
    print("Cleaned sentinel count:", dev_clean[col].isin([96, 98]).sum())
    print("Indicator count:", dev_clean[f"{col}_sentinel"].sum())


NumberOfTime30-59DaysPastDueNotWorse
Original sentinel count: 214
Cleaned sentinel count: 0


KeyError: 'NumberOfTime30-59DaysPastDueNotWorse_sentinel'

In [ ]:
print("Original missing values:")
print(dev.isnull().sum()[dev.isnull().sum() > 0])

print("\nCleaned missing values:")
print(dev_clean.isnull().sum()[dev_clean.isnull().sum() > 0])

Original missing values:
MonthlyIncome         23675
NumberOfDependents     3128
dtype: int64

Cleaned missing values:
age                                         1
NumberOfTime30-59DaysPastDueNotWorse      214
MonthlyIncome                           23675
NumberOfTimes90DaysLate                   214
NumberOfTime60-89DaysPastDueNotWorse      214
NumberOfDependents                       3128
dtype: int64


In [ ]:
from src.preprocessing import PercentileCapper

cap_columns = [
    "RevolvingUtilizationOfUnsecuredLines",
    "DebtRatio"
]

capper = PercentileCapper(columns=cap_columns)

# Fit only on development training data
capper.fit(dev_clean)

print("Learned upper limits:")
print(capper.upper_limits_)

Learned upper limits:
{'RevolvingUtilizationOfUnsecuredLines': np.float64(1.0944386634299974), 'DebtRatio': np.float64(4977.0)}


In [ ]:
dev_capped = capper.transform(dev_clean)

for col in cap_columns:
    print(f"\n{col}")
    print("Before capping:", dev_clean[col].max())
    print("After capping:", dev_capped[col].max())
    print("Learned limit:", capper.upper_limits_[col])


RevolvingUtilizationOfUnsecuredLines
Before capping: 50708.0
After capping: 1.0944386634299974
Learned limit: 1.0944386634299974

DebtRatio
Before capping: 329664.0
After capping: 4977.0
Learned limit: 4977.0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from src.preprocessing import make_pipeline

X = dev.drop(columns=["SeriousDlqin2yrs"])
y = dev["SeriousDlqin2yrs"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline = make_pipeline(
    LogisticRegression(max_iter=1000),
    scale=True
)

# Fit only on training data
pipeline.fit(X_train, y_train)

# Predict validation data
predictions = pipeline.predict(X_val)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Predictions generated:", len(predictions))
print("Pipeline verification successful!")

Training rows: 96000
Validation rows: 24000
Predictions generated: 24000
Pipeline verification successful!


In [ ]:
from src.evaluation import classification_metrics, plot_roc_pr

print("Evaluation utilities imported successfully!")

Evaluation utilities imported successfully!


In [ ]:
from src.evaluation import run_cv

print("CV runner imported successfully!")

CV runner imported successfully!


In [ ]:
import numpy as np
from src.evaluation import classification_metrics

y_true = np.array([0, 0, 1, 1, 0, 1])
y_prob = np.array([0.1, 0.2, 0.8, 0.9, 0.3, 0.7])

metrics = classification_metrics(y_true, y_prob)

print(metrics)

{'roc_auc': 1.0, 'pr_auc': 1.0, 'brier': 0.04666666666666667}


In [ ]:
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from src.evaluation import run_cv

# Generate dummy classification data
X_dummy, y_dummy = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)

# Create a simple dummy pipeline
dummy_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DummyClassifier(strategy="prior"))
])

# Run cross-validation
scores, summary = run_cv(dummy_pipeline, X_dummy, y_dummy)

print("Cross-validation results:")
print(summary)

Cross-validation results:
{'roc_auc_mean': np.float64(0.5), 'roc_auc_std': np.float64(0.0), 'pr_auc_mean': np.float64(0.502), 'pr_auc_std': np.float64(0.0040000000000000036), 'brier_mean': np.float64(0.250005), 'brier_std': np.float64(2.499999999999725e-06)}
